## #2a Neural Patterning

## Purpose: 

Explore neural patterning along the A-P and D-V axes and determine when spatial bias emerges during development.

## Setup

In [ ]:
import treedata as td
import pycea as py
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.colors as mcolors

from devmap.config import set_theme, stage_palette, lineage_palette, get_paths
from devmap.utils import save_plot, load_data
from devmap.config import lineage_palette, sequential_cmap, lineage_palette, stage_palette, colors, cooc_cmap, celltype_palette, subtype_palette
from devmap.progenitors import get_fate_progenitors, compute_pmi
from devmap.fate import extant_node_attribute
from devmap.plots import area_plot, barplot_with_points
from devmap.topology import nearest_internal_neighbors

set_theme()
base_path, plots_path, results_path = get_paths("axial_patterning")

## Load data

In [ ]:
tdata = load_data("log1p",scvi = True)
spatial_scores = pd.read_csv(results_path / "spatial_scores.csv", index_col=0)
tdata.obs["ap_score"] = spatial_scores["ap_score"]
tdata.obs["dv_score"] = spatial_scores["dv_score"]

## Identify neural progenitors

In [ ]:
tdata.obs["is_neural"] = tdata.obs["lineage"].isin(["Neural ectoderm", "Neural crest"]).astype(int)
py.tl.ancestral_states(tdata, "is_neural", method = "mean")
tdata.obs["is_neural"] = tdata.obs["is_neural"].astype(bool)
progenitors = get_fate_progenitors(tdata, "is_neural", key_added="neural_progenitors", min_descendants=1)

## Plot neural progenitors

Example clone

In [ ]:
fig, ax = plt.subplots(figsize=(1.2, 3.6), dpi = 600)
clone_tdata = tdata[tdata.obs.query("clone == 'E9.5-R2-C1' & is_neural == True").index,[]].copy()
nodes = clone_tdata.obs.query("clone == @clone & progenitor.notnull()").progenitor.unique().tolist()
py.pl.tree(clone_tdata, depth_key="time", tree = clone, ax = ax, branch_linewidth=.5)
py.pl.nodes(clone_tdata,tree = clone, nodes = nodes, ax = ax, color = colors[1], size = 5, outline_width=.3)
py.pl.annotation(clone_tdata, keys = "ap_score", cmap = "Oranges_r", label = False, ax = ax, gap = .05, legend = True, width = .15, vmax = 1, vmin = 0)
py.pl.annotation(clone_tdata, keys = "dv_score", cmap = "Purples_r", label = False, ax = ax, gap = .05, legend = True, width = .15, vmax = 1, vmin = 0, na_color = "lightgray")
save_plot(plots_path / "neural_progenitor_tree.svg", fig, rasterize=True)

Example clades

In [ ]:
for node in ["JMsYAeBf", "S3YBGu3Q","WoiheAKZ"]:
    clade_tdata = tdata[clone_tdata.obs.progenitor == node,[]]
    fig, ax = plt.subplots(figsize=(1.5,clade_tdata.shape[0]/20), dpi=600)
    py.pl.tree(clade_tdata, depth_key="time", ax = ax, branch_linewidth=0.5)
    py.pl.annotation(clade_tdata, keys = "cell_type", palette = subtype_palette, label = False, ax = ax, gap = .05, legend = True, width = .15)
    py.pl.annotation(clade_tdata, keys = "ap_score", cmap = "Oranges_r", label = False, ax = ax, gap = .05, legend = True, width = .15, vmax = 1, vmin = 0)
    py.pl.annotation(clade_tdata, keys = "dv_score", cmap = "Purples_r", label = False, ax = ax, gap = .05, legend = True, width = .15, vmax = 1, vmin = 0, na_color = "lightgray")
    save_plot(plots_path / f"neural_prog_tree_{node}.svg", fig)

## Region progenitor stats

In [ ]:
use_stages = ["E8.5","E9.0","E9.5"]
prog_stats_by_type = (
    tdata.obs.query("stage.isin(@use_stages) & neural_progenitor.notna()")
    .groupby(["stage","cell_type","embryo"], observed=True)
    .agg(
        mean_time=("prog_time", "mean"),
        mean_size=("prog_size", "mean"),
        n_progenitors=("neural_progenitor","nunique"),
    )
    .reset_index()
)
prog_stats_by_type["stage"] = prog_stats_by_type["stage"].astype(str)
order = [
    "Telencephalon","Diencephalon","Midbrain","Hindbrain (r1)",
    "Hindbrain (r2-r4)","Hindbrain (r5-r8)",
    "Spinal cord (cervical)","Spinal cord (thoracic)",
    "Spinal cord (lumbar)","Spinal cord (sacral)"
]

Clone size

In [ ]:
fig, ax = plt.subplots(figsize=(.9, 2), dpi=600)

barplot_with_points(
    prog_stats_by_type, x="mean_size", y="cell_type", hue="stage", order=order,
    hue_order=["E8.5", "E9.0", "E9.5"], palette=stage_palette, ax=ax,
    point_size=2, jitter=0.15)

ax.axvline(0, color="black", linestyle="-", linewidth=0.8)
ax.set_ylabel("")
ax.set_xlabel("Mean clone size")

save_plot(plots_path / "prog_size_by_neural_cell_type.svg", fig, rasterize=False)

Number of progenitors

In [ ]:
fig, ax = plt.subplots(figsize=(.9, 2), dpi=600)

barplot_with_points(
    prog_stats_by_type, x="n_progenitors", y="cell_type", hue="stage", order=order,
    hue_order=["E8.5", "E9.0", "E9.5"], palette=stage_palette, ax=ax,
    point_size=2, jitter=0.15)

ax.axvline(0, color="black", linestyle="-", linewidth=0.8)
ax.set_ylabel("")
ax.set_xlabel("Mean progenitor count")

save_plot(plots_path / "prog_count_by_neural_cell_type.svg", fig, rasterize=False)

Fate restriction time

In [ ]:
fig, ax = plt.subplots(figsize=(.9, 2), dpi=600)

sns.boxplot(data = barplot_with_points, x = "mean_time", y = "cell_type", 
            hue = "stage", palette=stage_palette, saturation = 1, ax=ax,
            linewidth=0.5, linecolor="black", order = order, legend=False)
sns.stripplot(
    data=barplot_with_points, x = "mean_time", y = "cell_type",
    dodge=True,jitter=0.15,color="black",size=2,legend=False,
    ax=ax,order = order, hue = "stage", palette=["black"]*3 
)
ax.set_ylabel("")
ax.set_xlabel("Mean progenitor time")

save_plot(plots_path / "prog_time_by_neural_cell_type.svg", fig, rasterize=False)

## Progenitor spatial extent heatmaps

AP and DV bins

In [ ]:
edges = np.linspace(0, .85, 18)
tdata.obs["ap_bin"] =  pd.cut(tdata.obs["ap_score"], edges, 
    labels=edges[:-1].round(2))

edges =np.linspace(.2, .8, 13)
tdata.obs["dv_bin"] =  pd.cut(tdata.obs["dv_score"], edges, 
    labels=edges[:-1].round(2))

Plot heatmaps

In [ ]:
for axis, ymax in [("ap", 2000), ("dv", 400)]:

    query = "is_neural & stage == 'E9.5' & dv_score.notnull()" if axis == "dv" else "is_neural & stage == 'E9.5'"
    large_clones = (tdata.obs.query(query).neural_progenitor.value_counts().loc[lambda x: x > 10].index)
    counts = (tdata.obs.query(f"{query} & neural_progenitor in @large_clones")
        .groupby(["neural_progenitor", f"{axis}_bin"]).size().unstack().fillna(0))
    frac = counts.div(counts.sum(axis=1), axis=0)

    positions = np.linspace(0, 1, counts.shape[1])
    frac["center_of_mass"] = (frac.values * positions).sum(axis=1)
    frac_sorted = frac.sort_values("center_of_mass").drop(columns=["center_of_mass"])
    total = counts.sum(axis=1).loc[frac_sorted.index]

    fig, axes = plt.subplots(2, 1, figsize=(2, 2), dpi=600,
        gridspec_kw={"height_ratios": [1, 3]},sharex=True)

    sns.barplot(
        x=np.arange(len(total)),
        y=total.values,
        ax=axes[0],
        color="black",
        saturation=1,
    )
    sns.heatmap(
        frac_sorted.T,
        cmap=sequential_cmap,
        yticklabels=False,
        ax=axes[1],
        cbar=False,
        vmax=.5,
    )

    axes[1].set_xticks(np.arange(len(total)) + 0.5)
    axes[1].set_xticklabels("", rotation=90)
    axes[0].set_xticks([])
    axes[0].set_ylim(0, ymax)
    plt.subplots_adjust(hspace=0)
    axes[1].set_ylabel(f"{axis.upper()} score")
    axes[0].set_ylabel("Total\noutput")
    axes[1].set_xlabel(f"Neural progenitors sorted by {axis.upper()} bias")

    save_plot(plots_path / f"neural_{axis}_heatmap.svg",fig,rasterize=True)

## Number of region represented

In [ ]:
df = (
    tdata.obs
    .query("stage == 'E9.5' & ap_score.notnull() & cell_type in @order")
    .groupby("progenitor")
    .agg(ap_mean=("ap_score", "mean"),size = ("ap_score", "size"),n_types = ("cell_type", "nunique"))
)

plt.figure(figsize=(2.1, 1.8), dpi=600)
fig, ax = plt.subplots(figsize=(2.1, 1.8), dpi=600)
plt.scatter(df, x = "time", y = "size", c = "n_types", cmap = "hot", alpha=0.5, s=5, linewidth=0, ax=ax)
plt.yscale("log")
plt.xlabel("time")
plt.ylabel("size")

save_plot(plots_path / "progenitor_size_vs_time_colored_by_n_types.svg", fig, rasterize=True)

## Progenitor co-ocurance

In [ ]:
order = ["Telencephalon","Diencephalon","Midbrain","Hindbrain (r1)","Hindbrain (r2-r4)","Hindbrain (r5-r8)",
        "Spinal cord (cervical)","Spinal cord (thoracic)","Spinal cord (lumbar)","Spinal cord (sacral)"]
cooc, pmi, npmi = compute_pmi(tdata.obs.query("stage == 'E9.5' & cell_type.isin(@order)"), group_col="neural_progenitor", cat_col="cell_type")
fig, ax = plt.subplots(figsize=(2,2), dpi=600)
sns.heatmap(pmi.loc[order, order], square=True, xticklabels=order, 
            yticklabels=order, cmap = cooc_cmap, center = 0, vmin = -2, vmax = 2, ax = ax)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)
save_plot(plots_path / "neural_cooc_heatmap.svg", fig, rasterize=False)

## Midbrain-hindbrain boundary UMAP

UMAP

In [ ]:
boundary_tdata = tdata[tdata.obs["cell_type"].isin(["Midbrain","Hindbrain (r1)",
             "Hindbrain (isthmic organizer)","Hindbrain (r2-r4)"]) & (tdata.obs["stage"] == 'E9.5')].copy()
sc.pp.neighbors(boundary_tdata, use_rep="X_scvi")
sc.tl.umap(boundary_tdata)

Colored by cell type

In [ ]:
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
sc.pl.umap(boundary_tdata, color = "cell_type", palette = celltype_palette, frameon=False, ax = ax, title="")
save_plot(plots_path / "midbrain_umap.svg", fig, rasterize=True)

Colored by marker gene

In [ ]:
for marker in ["Fgf8", "En1", "Otx2", "Hoxa2"]:
    fig, ax = plt.subplots(figsize=(3,3), dpi=600)
    sc.pl.umap(boundary_tdata, color = marker, cmap="viridis", frameon=False, ax = ax, title="", vmax = 2)
    save_plot(plots_path / f"midbrain_umap_{marker}.svg", fig, rasterize=True)

Color by neural progenitor

In [ ]:
fig, ax = plt.subplots(figsize=(3,3), dpi=600)
example_progs = boundary_tdata.obs.neural_progenitor.value_counts().sample(10).index.to_list()
sc.pl.umap(boundary_tdata, ax = ax, title = "", frameon=False, show = False) 
sc.pl.umap(boundary_tdata[boundary_tdata.obs.neural_progenitor.isin(example_progs)], s = 100,
    color = "neural_progenitor", groups = example_progs, ax = ax, title = "", frameon=False, palette = colors)
save_plot(plots_path / "midbrain_umap_progenitors.svg", fig, rasterize=True)

## Midbrain-hindbrain relative LCA

In [ ]:
py.tl.ancestral_linkage(boundary_tdata,"cell_type",target = "Hindbrain (r2-r4)", depth_key = "time")
py.tl.ancestral_linkage(boundary_tdata,"cell_type",target = "Midbrain", depth_key = "time")
boundary_tdata.obs["Midbrain_lca"] = 9.5 - boundary_tdata.obs["Midbrain_linkage"]/2
boundary_tdata.obs["r2_lca"] = 9.5 - boundary_tdata.obs["Hindbrain (r2-r4)_linkage"]/2
boundary_tdata.obs["delta_lca"] = boundary_tdata.obs["r2_lca"] - boundary_tdata.obs["Midbrain_lca"]

Region LCA

In [ ]:
for region, cmap in [("Midbrain", "Blues"), ("r2", "Reds")]:
    fig, ax = plt.subplots(figsize=(3.2, 3), dpi=600)
    sc.pl.umap(
        boundary_tdata[boundary_tdata.obs.delta_linkage.notnull()],
        color=f"{region}_lca",cmap=cmap,sort_order=False,frameon=False,
        title="",vmin=6,ax=ax)
    save_plot(plots_path / f"midbrain_umap_{region.lower()}_lca.svg",fig,rasterize=True,)

Relative LCA

In [ ]:
fig, ax = plt.subplots(figsize=(3.2,3), dpi=600)
sc.pl.umap(boundary_tdata[boundary_tdata.obs.delta_lca.notnull()], color="delta_lca", 
    cmap = "RdBu_r", vmin=-3, vmax=3, sort_order=False, frameon=False, ax = ax, title="")
save_plot(plots_path / "midbrain_umap_delta_lca.svg", fig, rasterize=True)

Relave LCA violin plot

In [ ]:
df = boundary_tdata.obs.query("cell_type.isin(['Hindbrain (r1)','Hindbrain (isthmic organizer)'])").copy()
df.cell_type = df.cell_type.cat.remove_unused_categories()
fig, ax = plt.subplots(figsize=(1,1), dpi=600)
sns.violinplot(data = df, x = "cell_type", y = "delta_linkage", 
    color="lightgray", bw_adjust=2,linewidth=0.8, edgecolor="black")
ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha="right")
plt.ylim(-5, 5)
save_plot(plots_path / "midbrain_delta_linkage_violin.svg", fig, rasterize=False)

## Hindbrain r1 fate restriction

Get potential of r1 ancestors

In [ ]:
boundary_tdata.obs["is_r1"] = (boundary_tdata.obs["region"] == "Hindbrain (r1)").astype(int)
boundary_tdata.obs["is_r2"] = (boundary_tdata.obs["region"] == "Hindbrain (r2-r4)").astype(int)
boundary_tdata.obs["is_mid"] = (boundary_tdata.obs["region"] == "Midbrain").astype(int)
py.tl.ancestral_states(boundary_tdata, "is_r1", method = "mean")
py.tl.ancestral_states(boundary_tdata, "is_r2", method = "mean")
py.tl.ancestral_states(boundary_tdata, "is_mid", method = "mean")
r1_paths = extant_node_attribute(boundary_tdata[boundary_tdata.obs["is_r1"] == 1],key = ["is_r1", "is_r2", "is_mid"], depth_key="time",paths = True,
                           bins = np.arange(0, 10, .5), return_type = "dataframe")
r1_paths["embryo"] = r1_paths.tree.str.split("-C").str[0]
conditions = [
    (r1_paths["is_r2"] == 0) & (r1_paths["is_mid"] == 0),
    (r1_paths["is_r2"] > 0) & (r1_paths["is_mid"] == 0),
    (r1_paths["is_r2"] == 0) & (r1_paths["is_mid"] > 0),
    (r1_paths["is_r2"] > 0) & (r1_paths["is_mid"] > 0),
]
choices = ["r1_only", "r1,r2", "r1,mid", "all"]
r1_paths["prog_type"] = np.select(conditions, choices, default="unknown")

Plot progenitor potential over time

In [ ]:
prog_type_palette = {
    "r1_only": celltype_palette["Hindbrain (r1)"],
    "r1,r2": "purple",
    "r1,mid": "orange",
    "all": "lightgray",
}
fig, ax = plt.subplots(figsize=(1.5,1.5), dpi=600)
df = r1_paths.groupby("time")["prog_type"].value_counts(normalize=True).reset_index(name = "count")
df["percent"] = df["count"] * 100
sns.lineplot(data=df, x="time", y="percent", hue="prog_type", palette = prog_type_palette, ax=ax)
plt.xticks([0,2,4,6,8,10]);
save_plot(plots_path / "r1_prog_types_over_time.svg", fig, rasterize=False)

In [ ]:
## Mark siblings
def leaves_below(G, node):
    descendants = nx.descendants(G, node)
    leaves = [n for n in descendants if G.out_degree(n) == 0]
    return leaves

def mark_sibling_descendants(tdata, progenitors, key_added = "sibling_descendants"):
    tdata.obs[key_added] = pd.NA
    for clone in progenitors["clone"].unique():
        tree = tdata.obst[clone]
        if len(tree) == 0:
            continue
        clone_progenitors = progenitors.query("clone == @clone")
        for i, prog in clone_progenitors.iterrows():
            if prog["qualified_via"] == 'bubble_up':
                sibling = prog["node"]
                sibling_leaves = leaves_below(tree, sibling)
            else:
                sibling = list(tree.predecessors(prog["node"]))[0]
                sibling_leaves = leaves_below(tree, sibling)
            if len(sibling_leaves) > 5 * prog["fate_descendants"]:
                    continue
            tdata.obs.loc[sibling_leaves, key_added] = prog["node"]

mark_sibling_descendants(tdata, progenitors)



progenitors["ap_bin"] = pd.cut(progenitors["ap_score"], edges, 
    labels=edges[:-1].round(2))
tdata.obs["sibling_ap"] = tdata.obs.sibling_descendants.map(progenitors["ap_bin"])



progenitors["dv_bin"] = pd.cut(progenitors["dv_score"], edges, 
    labels=edges[:-1].round(2))
tdata.obs["sibling_dv"] = tdata.obs.sibling_descendants.map(progenitors["dv_bin"])

## Heritability of spatial position

In [ ]:
nearest_neighbors = {}
for clone in tdata.obs.query("stage == 'E9.5' & clone.notnull()")["clone"].unique():
    tree = tdata.obst[clone].copy()
    nodes = progenitors.query("clone == @clone")["node"]
    nearest_neighbors.update(nearest_internal_neighbors(tree, nodes))

progenitors["ap_score"] = tdata.obs.groupby("progenitor")["ap_score"].mean()
progenitors["dv_score"] = tdata.obs.groupby("progenitor")["dv_score"].mean()
progenitors["nearest_neighbor"] = progenitors["node"].map(nearest_neighbors)
progenitors["nearest_neighbor_ap"] = progenitors["nearest_neighbor"].map(progenitors["ap_score"])
progenitors["nearest_neighbor_dv"] = progenitors["nearest_neighbor"].map(progenitors["dv_score"])

Spatial position scatter plot

In [ ]:
axis_colors = {"ap": "#fd8c3b", "dv": "#7262ac"}
for axis in ["ap", "dv"]:

    score = f"{axis}_score"
    neighbor = f"nearest_neighbor_{axis}"
    df = progenitors.query(f"{score}.notnull() and {neighbor}.notnull()").copy()

    fig, ax = plt.subplots(figsize=(1, 1), dpi=600)
    sns.scatterplot(
        data=df, x=score, y=neighbor, color=axis_colors[axis], s=.8,
        linewidth=0, alpha=.2, legend=False, ax=ax
    )
    lims = (0,.9) if axis == "ap" else (.2,.8)
    ax.plot(lims, lims, color="black", linestyle="--", linewidth=1, zorder=-1)
    ax.set_xticks(lims)
    ax.set_yticks(lims)
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    r2 = np.corrcoef(df[score], df[neighbor])[0, 1] ** 2
    ax.text(0.05, 0.85, f"$R^2$ = {r2:.2f}", transform=ax.transAxes, fontsize=8)

    ax.set_xlabel(f"{axis.upper()} score")
    ax.set_ylabel(f"Neighbor {axis.upper()}")

    save_plot(plots_path / f"nearest_neighbor_{axis}_correlation.svg", fig, rasterize=True)

## Progenitor time vs spatial score

In [ ]:
for axis in ["ap", "dv"]:
    df = progenitors.query("stage == 'E9.5'")
    fig, ax = plt.subplots(figsize=(1.8,1.8), dpi=600)
    sns.scatterplot(data=df,x="time",y=f"{axis}_score",color = axis_colors[axis],
        s=3,linewidth=0,alpha = .2,legend=False,ax=ax)
    plt.xlim(5,10)
    r2 = np.corrcoef(df[f"{axis}_score"], df["time"])[0,1] ** 2
    ax.text(0.05, 0.85, f"$R^2$ = {r2:.2f}", transform=ax.transAxes, fontsize=8)
    save_plot(plots_path / f"{axis}_score_time_correlation.svg", fig, rasterize=True)

## Spatial score over time

Infer positions of ancestral nodes

In [ ]:
py.tl.ancestral_states(tdata, keys = f"ap_score", method = "mean")
py.tl.ancestral_states(tdata, keys = f"dv_score", method = "mean")
paths = extant_node_attribute(tdata[tdata.obs["stage"] == "E9.5"],key = [f"ap_score","dv_score"], depth_key="time",paths = True,
                           bins = np.arange(0, 10, .5), return_type = "dataframe", sample=10000)

Ancestral path lineplot

In [ ]:
for axis in ["ap", "dv"]:
    paths["leaf_bin"] = paths.leaf.map(tdata.obs[f"{axis}_bin"])
    df = paths.groupby(["leaf_{axis}_bin","time"])["{axis}_score"].mean().reset_index()
    fig, ax = plt.subplots(figsize=(1.3,1), dpi=600)
    palette = sns.color_palette("Oranges", n_colors=20) if axis == "ap" else sns.color_palette("Purples", n_colors=20)
    sns.lineplot(data=df, x="time", y="{axis}_score", hue="leaf_{axis}_bin", palette=palette, 
                legend = False, ax = ax, linewidth=0.8)
    plt.xticks([0,5,10])
    save_plot(plots_path / f"{axis}_score_ancestral_paths.svg", fig, rasterize=False)